# 0. Preparation

In [1]:
import os
import json
import pandas as pd
import numpy as np
from collections import defaultdict, Counter
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
import matplotlib.pyplot as plt


In [2]:
data_path = os.path.join("..", "..", "data", "processed_dataset_v2.csv")
df = pd.read_csv(data_path, engine="python", on_bad_lines="skip")
df.head()

,pid,url,processed_text,title,description,brand_facet,category_facet,subcategory_facet,seller_facet,discount,selling_price,actual_price,average_rating,attributes
0,TKPFCZ9EA7H5FYZH,https://www.flipkart.com/yorker-solid-men-mult...,"['solid', 'women', 'multicolor', 'track', 'pan...",Solid Women Multicolor Track Pants,Yorker trackpants made from 100% rich combed c...,york,clothing_and_accessories,bottomwear,shyam_enterprises,69.0,921.0,2999.0,3.9,"['elast', 'side', 'pocket', 'cotton', 'blend',..."
1,TKPFCZ9EJZV2UVRZ,https://www.flipkart.com/yorker-solid-men-blue...,"['solid', 'men', 'blue', 'track', 'pant', 'yor...",Solid Men Blue Track Pants,Yorker trackpants made from 100% rich combed c...,york,clothing_and_accessories,bottomwear,shyam_enterprises,66.0,499.0,1499.0,3.9,"['drawstr', 'elast', 'side', 'pocket', 'cotton..."
2,TKPFCZ9EHFCY5Z4Y,https://www.flipkart.com/yorker-solid-men-mult...,"['solid', 'men', 'multicolor', 'track', 'pant'...",Solid Men Multicolor Track Pants,Yorker trackpants made from 100% rich combed c...,york,clothing_and_accessories,bottomwear,shyam_enterprises,68.0,931.0,2999.0,3.9,"['elast', 'side', 'pocket', 'cotton', 'blend',..."
3,TKPFCZ9ESZZ7YWEF,https://www.flipkart.com/yorker-solid-men-mult...,"['solid', 'women', 'multicolor', 'track', 'pan...",Solid Women Multicolor Track Pants,Yorker trackpants made from 100% rich combed c...,york,clothing_and_accessories,bottomwear,shyam_enterprises,69.0,911.0,2999.0,3.9,"['elast', 'side', 'pocket', 'cotton', 'blend',..."
4,TKPFCZ9EVXKBSUD7,https://www.flipkart.com/yorker-solid-men-brow...,"['solid', 'women', 'brown', 'gray', 'track', '...","Solid Women Brown, Grey Track Pants",Yorker trackpants made from 100% rich combed c...,york,clothing_and_accessories,bottomwear,shyam_enterprises,68.0,943.0,2999.0,3.9,"['drawstr', 'elast', 'side', 'pocket', 'cotton..."


In [6]:
# we need all the tokens in a single column for the inverted index, hence we concatenate the processed_text and attributes columns
df['tokens'] = df['processed_text'] + df['attributes']

# 1. Ranking

## 1.1 TF-IDF + cosine similarity

Since we already implemented this ranking method in part 2, we will reuse the code in order to later compare with BM25 and our own ranking method. 

In [7]:
with open(os.path.join("..", "..", "project_progress", "part_2", "inverted_index.json"), "r") as f:
    inverted_index = json.load(f)

with open(os.path.join("..", "..", "project_progress", "part_2", "idf_scores.json"), "r") as f:
    idf_scores = json.load(f)

In [8]:
def setup_preprocessing_tools():
    stemmer = PorterStemmer()
    stop_words = set(stopwords.words("english"))
    stop_words.update(['made', 'wear', 'comfort', 'quality', 
                       'look', 'perfect', 'style', 'great', 'cool'])
    stop_words.discard('no')
    stop_words.discard('not')
    return stemmer, stop_words

In [12]:
def preprocess_query(text, stemmer, stop_words):
    """
    Preprocess natural text:
    - lowercase
    - remove punctuation/numbers
    - tokenize
    - remove stopwords and non-alphabetic tokens
    - stem 
    """
    if not isinstance(text, str):
        return []
    
    text = text.replace('-', ' ')

    # lowercase
    text = text.lower()

    # remove punctuation and digits
    text = re.sub(f"[{re.escape(string.punctuation)}0-9]", " ", text)

    # tokenize
    tokens = word_tokenize(text)

    # filter tokens (stopwords, non-alpha, short tokens)
    tokens = [w for w in tokens if w.isalpha() and w not in stop_words]

    # stem
    tokens = [stemmer.stem(w) for w in tokens]

    # normalize color terms 
    color_map = {'navy': 'blue', 'grey': 'gray', 'fucsia': 'pink', 'burgundy': 'red', 'violet': 'purple', 'beige': 'brown', 'magenta': 'pink', 'indigo': 'blue', 
                 'charcoal': 'gray', 'crimson': 'red', 'teal': 'green', 'lavender': 'purple', 'mustard': 'yellow', 'turquoise': 'blue', 'peach': 'orange'}

    tokens = [color_map.get(w, w) for w in tokens]

    return tokens

In [9]:
def calculate_log_tf(tokens):
    """Calculates log-normalized TF for each term in a document's token list."""
    counts = Counter(tokens)
    return {term: 1 + np.log(count) for term, count in counts.items()} #Don't need to handle log(0) since count is always >=1 because it's only done for terms that appear in tokens

# Apply this function to every document's tokens
df['tf_scores'] = df['tokens'].apply(calculate_log_tf)

# Check the TF scores for the first document
print("TF scores for first document:")
print(df.iloc[0]['tf_scores'])

TF scores for first document:
{'[': np.float64(1.6931471805599454), "'": np.float64(5.127134385045092), 's': np.float64(3.0794415416798357), 'o': np.float64(3.772588722239781), 'l': np.float64(3.302585092994046), 'i': np.float64(3.8903717578961645), 'd': np.float64(3.302585092994046), ',': np.float64(4.367295829986475), ' ': np.float64(4.367295829986475), 'w': np.float64(1.6931471805599454), 'm': np.float64(2.386294361119891), 'e': np.float64(3.5649493574615367), 'n': np.float64(3.4849066497880004), 'u': np.float64(2.6094379124341005), 't': np.float64(3.6390573296152584), 'c': np.float64(3.4849066497880004), 'r': np.float64(3.6390573296152584), 'a': np.float64(3.302585092994046), 'k': np.float64(2.6094379124341005), 'p': np.float64(2.386294361119891), 'y': np.float64(1.6931471805599454), 'h': np.float64(2.09861228866811), 'b': np.float64(2.386294361119891), 'g': np.float64(1.6931471805599454), 'v': np.float64(1.0), 'f': np.float64(2.09861228866811), ']': np.float64(1.6931471805599454)}

In [10]:
def calculate_tfidf_L2_norm(tf_scores, idf_scores_global):
    """
    Calculates the TF-IDF vector (as a dict) and the L2-norm (length)
    of that vector for a single document.
    """
    tfidf_vector = {}
    sum_of_squares = 0.0
    
    for term, tf in tf_scores.items():
        # Only include terms that are in our global IDF dictionary
        if term in idf_scores_global:
            tfidf = tf * idf_scores_global[term] # we multiply each TF value by the global IDF score
            tfidf_vector[term] = tfidf
            sum_of_squares += tfidf**2 
            
    doc_length = np.sqrt(sum_of_squares) # we compute the L2 norm 
    return tfidf_vector, doc_length

# Apply the function to the 'tf_scores' column
# This returns a tuple (tfidf_vector, doc_length), so we split it into two new columns
tfidf_results = df['tf_scores'].apply(lambda tf: calculate_tfidf_L2_norm(tf, idf_scores))
df['tfidf_vector'] = tfidf_results.apply(lambda x: x[0]) # tfidf vector dictionary
df['doc_length'] = tfidf_results.apply(lambda x: x[1]) # document length

# Check the results for the first document
print("TF-IDF vector for first document:")
print(df.iloc[0]['tfidf_vector'])
print("\nDocument length for first document:")
print(df.iloc[0]['doc_length'])

TF-IDF vector for first document:
{'l': np.float64(7.218750165346522), 'w': np.float64(11.826213040609899), 'e': np.float64(22.101963052398602), 'n': np.float64(18.841379768232994), 'u': np.float64(14.128892209800249), 'c': np.float64(17.44685932852257), 'r': np.float64(19.214236673553), 'k': np.float64(15.641892573643768), 'p': np.float64(16.24794724364417), 'h': np.float64(13.328345767290966), 'b': np.float64(15.013620940803683), 'g': np.float64(8.432140952448298), 'v': np.float64(3.8593418351572426), 'f': np.float64(11.329697121944546)}

Document length for first document:
55.08852133163933


In [11]:
stemmer, stop_words = setup_preprocessing_tools()
df_indexed = df.set_index('pid')

def search_tfidf(query_text, inverted_index, idf_scores, df_docs, k=10):
    """
    Performs a ranked TF-IDF search for a given query.
    
    Args:
        query_text (str): The raw query string.
        inverted_index (dict): The inverted index.
        idf_scores (dict): The pre-calculated IDF scores for all terms.
        df_docs (pd.DataFrame): The DataFrame (indexed by 'pid') 
                                containing 'tfidf_vector' and 'doc_length'.
        k (int): The number of top results to return.

    Returns:
        list: A list of (pid, score) tuples, sorted by score.
    """
    
    # Preprocess the query
    query_tokens = preprocess_query(query_text, stemmer, stop_words)
    
    # Find matching documents (Conjunctive/AND query)
    try:
        # Retrieve the set of pids for each query token
        doc_sets = [set(inverted_index[token]) for token in query_tokens if token in inverted_index]
        
        # If any token is not in the index, no docs will match an AND query
        if len(doc_sets) != len(query_tokens):
            print("One or more query terms not in index. No results.")
            return []

        # Find the intersection of all sets
        matching_pids = set.intersection(*doc_sets)
    
    except KeyError:
        # This handles if a token isn't in the index, though the check above is safer
        print("Query term not in index. No results.")
        return []

    if not matching_pids:
        print("No documents contain all query terms.")
        return []

    # Calculate the Query TF-IDF Vector and its length
    query_tf = calculate_log_tf(query_tokens)
    query_tfidf_vector = {}
    query_sum_of_squares = 0.0
    
    for term, tf in query_tf.items():
        if term in idf_scores:
            tfidf = tf * idf_scores[term]
            query_tfidf_vector[term] = tfidf
            query_sum_of_squares += tfidf**2
            
    query_length = np.sqrt(query_sum_of_squares)
    
    if query_length == 0:
        print("Query vector has no length (all terms unknown).")
        return []

    # Calculate Cosine Similarity for all matching documents
    scores = {}
    
    # Filter the main DataFrame to only the documents that matched
    # .loc is fast because we set the index to 'pid'
    matching_docs = df_docs.loc[list(matching_pids)]
    
    for pid, row in matching_docs.iterrows():
        doc_tfidf_vector = row['tfidf_vector']
        doc_length = row['doc_length']
        
        # Calculate Dot Product
        dot_product = 0.0
        # Iterate over the query vector, which is much smaller
        for term, query_tfidf_val in query_tfidf_vector.items():
            if term in doc_tfidf_vector:
                dot_product += query_tfidf_val * doc_tfidf_vector[term]
        
        # Calculate Cosine Similarity
        if doc_length > 0:
            scores[pid] = dot_product / (query_length * doc_length)
    
    # Sort and return the top K results
    sorted_results = sorted(scores.items(), key=lambda item: item[1], reverse=True)
    
    final_results = []
    # Loop through the top k sorted (pid, score) tuples
    for pid, score in sorted_results[:k]:
        
        # Get the product data
        product_data = df_docs.loc[pid]
        
        # Append a dictionary with all the info we want
        final_results.append({
            'title': product_data['title'],
            'score': score,
            'pid': pid,
            'url': product_data['url'],
            'selling_price': product_data['selling_price'],
            'brand': product_data['brand_facet']
        })
        
    return final_results

## 1.2 BM25

In [16]:
# Compute document lengths and average length
df['L_d'] = df['tokens'].apply(len)
L_ave = df['doc_len'].mean()
N = len(df)
print(f"Number of documents: {N}, Average length: {L_ave:.2f}")

Number of documents: 28081, Average length: 396.56


In [ ]:
df_counts = {term: len(postings) for term, postings in inverted_index.items()}

def bm25_idf(df_t, N):
    return np.log((N - df_t + 0.5) / (df_t + 0.5) + 1)

bm25_idf_scores = {t: bm25_idf(df_t, N) for t, df_t in df_counts.items()}


In [ ]:
def search_bm25(query_text, inverted_index, bm25_idf, df_docs,
                k=10, k1=1.5, b=0.75):
    """
    BM25 ranking for a given query.

    Args:
        query_text (str): Raw query string.
        inverted_index (dict): Inverted index (term -> list of pids).
        bm25_idf (dict): Precomputed BM25 IDF scores.
        df_docs (pd.DataFrame): DataFrame indexed by 'pid', containing 'tokens' and 'doc_len'.
        k (int): Number of top results to return.
        k1, b (float): BM25 hyperparameters.

    Returns:
        List[dict]: Ranked list of top-k results with pid, title, brand, etc.
    """

    # 1) Preprocess query 
    query_tokens = preprocess_query(query_text, stemmer, stop_words)
    if not query_tokens:
        return []

    # 2) Candidate documents: union of postings of query terms 
    doc_sets = [set(inverted_index[t]) for t in query_tokens if t in inverted_index]
    if not doc_sets or len(doc_sets) < len(query_tokens):
        return []
    candidate_pids = set.intersection(*doc_sets)
    if not candidate_pids:
        return []

    # 3) Compute BM25 scores for candidate documents
    scores = {}
    L_ave = df_docs['doc_len'].mean()

    for pid in candidate_pids:
        row = df_docs.loc[pid]
        doc_tokens = row['tokens']
        Ld = row['doc_len']
        score = 0.0

        # term frequencies in this doc
        tf_doc = Counter(doc_tokens)

        for t in query_tokens:
            if t not in bm25_idf_scores:
                continue
            tf_td = tf_doc.get(t, 0)
            if tf_td == 0:
                continue

            idf = bm25_idf_scores[t]

            # BM25 term contribution
            denom = tf_td + k1 * ((1 - b) + b * (Ld / L_ave))
            term_score = idf * ((k1 + 1) * tf_td / denom)
            score += term_score

        if score > 0:
            scores[pid] = score

    if not scores:
        return []

    # --- Sort and format top-k results ---
    sorted_pids = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k]
    results = []
    for pid, s in sorted_pids:
        row = df_docs.loc[pid]
        results.append({
            "pid": pid,
            "score": float(s),
            "title": row.get("title", ""),
            "brand": row.get("brand_facet", ""),
            "url": row.get("url", ""),
            "selling_price": row.get("selling_price", np.nan)
        })
    return results
